# 논문 재현형 엄격 베이스라인

이 노트북은 논문 **"A Case Study on the Data Mining-Based Prediction of Students' Performance for Effective and Sustainable E-Learning"**의 예측 변수 구성을 OULAD baseline CSV에 맞춰 엄격하게 단순화한 베이스라인 실험이다.

입력 파일은 `data/processed/features_week{5,7,10}_baseline.csv`이다. 이 파일은 active-at-cutoff cohort, `target_at_risk`, VLE mean 중복 처리, 결측 플래그/채움, missing due-date assessment 제외가 적용된 Stage 5 산출물이다.

이 실험은 최종 모델링이 아니라, 논문 재현 관점의 기준 성능을 확인하기 위한 베이스라인이다. 따라서 학습 과정과 직접 관련된 두 가지 특징만 사용한다.

- `active_days_until_cutoff`: OULAD에는 직접적인 로그인 수가 없으므로, 컷오프 시점까지 활동한 날짜 수를 access/login proxy로 사용한다.
- `total_click_until_cutoff_mean_strategy`: VLE 중복 key의 `sum_click` 평균 전략을 적용한 컷오프 시점까지의 클릭 수를 click count 특징으로 사용한다.

인구통계 특징, 평가/assessment 특징, module/presentation 식별자, activity_type별 클릭 특징, missing flag, `id_student`, `final_result`, target 컬럼은 엄격 재현 베이스라인의 입력 특징에서 의도적으로 제외한다.


## 1. 실행 환경과 공통 설정

프로젝트 루트에서 실행되는 것을 전제로 경로를 설정한다. 필요한 폴더가 없으면 생성하되, 원본 데이터 파일은 수정하지 않는다.

In [19]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists() and (candidate / "reports").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/processed and reports")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "model_results"
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data dir exists: {DATA_DIR.exists()}")
print(f"Report table dir: {REPORT_TABLE_DIR}")

Project root: C:\Users\osca0\Dev\studyDataMining\teamproject
Processed data dir exists: True
Report table dir: C:\Users\osca0\Dev\studyDataMining\teamproject\reports\tables\model_results


## 2. 실험 대상 파일과 특징 정의

week 5, week 7, week 10 baseline 파일을 사용한다. VLE 중복 처리 방식은 Stage 5에서 mean strategy로 확정했으므로 이 노트북에서는 더 이상 max/sum strategy를 비교하지 않는다.

엄격 재현 베이스라인의 입력 특징은 정확히 두 개다. `code_module`, `code_presentation`, `id_student`, `final_result`, `target_withdrawn`, `target_at_risk` 컬럼은 입력 행렬 `X`에 넣지 않는다.


In [20]:
cutoff_files = {
    "week5": DATA_DIR / "features_week5_baseline.csv",
    "week7": DATA_DIR / "features_week7_baseline.csv",
    "week10": DATA_DIR / "features_week10_baseline.csv",
}

feature_set_name = "mean"
features = [
    "total_click_until_cutoff_mean_strategy",
    "active_days_until_cutoff",
]

target_name = "target_at_risk"
excluded_predictor_columns = {
    "code_module",
    "code_presentation",
    "id_student",
    "final_result",
    "target_withdrawn",
    "target_at_risk",
}

for cutoff, path in cutoff_files.items():
    print(f"{cutoff}: {path} | exists={path.exists()}")


week5: C:\Users\osca0\Dev\studyDataMining\teamproject\data\processed\features_week5_baseline.csv | exists=True
week7: C:\Users\osca0\Dev\studyDataMining\teamproject\data\processed\features_week7_baseline.csv | exists=True
week10: C:\Users\osca0\Dev\studyDataMining\teamproject\data\processed\features_week10_baseline.csv | exists=True


## 3. 타깃과 입력 검증 함수

주요 타깃은 Stage 5 baseline CSV에 이미 생성된 `target_at_risk`를 사용한다.

- `Withdrawn`, `Fail` -> 1, at-risk 양성 클래스
- `Pass`, `Distinction` -> 0, non-risk 음성 클래스

행을 조용히 제거하지 않기 위해, 필요한 컬럼 누락, `target_at_risk`와 `final_result`의 불일치, base key 중복, 결측값, 무한값이 있으면 즉시 오류를 발생시킨다.


In [21]:
target_map = {
    "Withdrawn": 1,
    "Fail": 1,
    "Pass": 0,
    "Distinction": 0,
}

base_key = ["code_module", "code_presentation", "id_student"]


def load_and_validate_baseline(path: Path, cutoff: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing baseline file for {cutoff}: {path}")

    df = pd.read_csv(path)
    required_columns = set(base_key) | {"final_result", target_name, "target_withdrawn", *features}

    missing_columns = sorted(required_columns - set(df.columns))
    if missing_columns:
        raise ValueError(f"{cutoff} is missing required columns: {missing_columns}")

    if "date_unregistration" in df.columns:
        raise ValueError(f"{cutoff} contains leakage column date_unregistration")

    duplicated_keys = int(df.duplicated(subset=base_key, keep=False).sum())
    if duplicated_keys:
        raise ValueError(f"{cutoff} has duplicated base key rows: {duplicated_keys}")

    unknown_results = sorted(set(df["final_result"].dropna().unique()) - set(target_map.keys()))
    if unknown_results:
        raise ValueError(f"{cutoff} has unexpected final_result values: {unknown_results}")

    if df["final_result"].isna().any():
        raise ValueError(f"{cutoff} has missing final_result values")

    expected_target = df["final_result"].map(target_map).astype(int)
    if not df[target_name].equals(expected_target):
        mismatch_count = int((df[target_name] != expected_target).sum())
        raise ValueError(f"{cutoff} target_at_risk mismatches final_result mapping: {mismatch_count}")

    if df[target_name].isna().any() or not set(df[target_name].unique()).issubset({0, 1}):
        raise ValueError(f"{cutoff} target_at_risk must be binary 0/1")

    missing_flag_columns = [col for col in df.columns if col.endswith("_missing")]
    non_binary_flags = [
        col for col in missing_flag_columns
        if not set(df[col].dropna().unique()).issubset({0, 1})
    ]
    if non_binary_flags:
        raise ValueError(f"{cutoff} has non-binary missing flags: {non_binary_flags}")

    print(f"\n[{cutoff}] rows: {len(df):,}")
    print("final_result distribution:")
    print(df["final_result"].value_counts().sort_index())
    print("target_at_risk distribution:")
    print(df[target_name].value_counts().sort_index())
    print(f"target_at_risk positive rate: {df[target_name].mean():.4f}")
    print(f"missing flag columns: {len(missing_flag_columns)}")

    return df


def build_xy(df: pd.DataFrame, features: list[str], cutoff: str):
    leaked_features = sorted(set(features) & excluded_predictor_columns)
    if leaked_features:
        raise ValueError(f"{cutoff} feature set includes excluded columns: {leaked_features}")

    missing_values = df[features].isna().sum()
    if missing_values.any():
        raise ValueError(
            f"{cutoff}/{feature_set_name} has missing feature values: "
            f"{missing_values[missing_values > 0].to_dict()}"
        )

    X = df.loc[:, features].copy()
    y = df[target_name].copy()

    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(f"{cutoff}/{feature_set_name} has non-finite feature values")

    return X, y


## 4. 모델과 교차검증 설정

논문 재현형 기준 성능 비교를 위해 Decision Tree, Gaussian Naive Bayes, Random Forest, SVC, KNN을 동일한 5-fold StratifiedKFold로 평가한다.

양성 클래스는 at-risk, 즉 `target_at_risk = 1`이다. precision, recall, f1은 모두 at-risk 클래스를 기준으로 계산한다.


In [22]:
models = {
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None,
    ),
    "SVC": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", random_state=RANDOM_STATE)),
        ]
    ),
    "KNeighborsClassifier": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier()),
        ]
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
}

print("Models:", list(models.keys()))
print("Scoring:", list(scoring.keys()))

Models: ['DecisionTreeClassifier', 'GaussianNB', 'RandomForestClassifier', 'SVC', 'KNeighborsClassifier']
Scoring: ['precision', 'recall', 'f1', 'roc_auc', 'accuracy']


## 5. 컷오프별 mean-strategy baseline 평가

각 cutoff에서 정확히 두 개의 특징만 사용한다. 이 단계는 additional feature engineering을 하지 않으며, 이미 생성된 baseline 파일의 두 learning-process feature만 입력으로 사용한다.


In [23]:
rows = []

for cutoff, path in cutoff_files.items():
    df = load_and_validate_baseline(path, cutoff)
    X, y = build_xy(df, features, cutoff)
    print(f"\nEvaluating {cutoff}/{feature_set_name} with features: {features}")

    for model_name, model in models.items():
        scores = cross_validate(
            clone(model),
            X,
            y,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=False,
        )

        result = {
            "cutoff": cutoff,
            "strategy": feature_set_name,
            "model": model_name,
            "n_rows": len(df),
            "positive_class": "at_risk",
            "target": target_name,
            "features": ",".join(features),
        }

        for metric in scoring:
            values = scores[f"test_{metric}"]
            result[f"{metric}_mean"] = values.mean()
            result[f"{metric}_std"] = values.std(ddof=1)

        rows.append(result)
        print(
            f"  {model_name}: "
            f"f1={result['f1_mean']:.4f}, "
            f"recall={result['recall_mean']:.4f}, "
            f"roc_auc={result['roc_auc_mean']:.4f}, "
            f"accuracy={result['accuracy_mean']:.4f}"
        )

results = pd.DataFrame(rows)
results



[week5] rows: 27,244
final_result distribution:
final_result
Distinction     3024
Fail            7044
Pass           12361
Withdrawn       4815
Name: count, dtype: int64
target_at_risk distribution:
target_at_risk
0    15385
1    11859
Name: count, dtype: int64
target_at_risk positive rate: 0.4353
missing flag columns: 12

Evaluating week5/mean with features: ['total_click_until_cutoff_mean_strategy', 'active_days_until_cutoff']
  DecisionTreeClassifier: f1=0.4877, recall=0.4840, roc_auc=0.5443, accuracy=0.5574
  GaussianNB: f1=0.6204, recall=0.7642, roc_auc=0.6720, accuracy=0.5929
  RandomForestClassifier: f1=0.5079, recall=0.5190, roc_auc=0.5808, accuracy=0.5621
  SVC: f1=0.5230, recall=0.4613, roc_auc=0.6584, accuracy=0.6337
  KNeighborsClassifier: f1=0.5042, recall=0.4819, roc_auc=0.6070, accuracy=0.5876

[week7] rows: 26,776
final_result distribution:
final_result
Distinction     3024
Fail            7044
Pass           12361
Withdrawn       4347
Name: count, dtype: int64
target

,cutoff,strategy,model,n_rows,positive_class,target,features,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std
0,week5,mean,DecisionTreeClassifier,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.491442,0.005863,0.484020,0.005825,0.487701,0.005741,0.544346,0.005252,0.557370,0.005025
1,week5,mean,GaussianNB,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.522189,0.004557,0.764229,0.005064,0.620422,0.003309,0.671995,0.003399,0.592938,0.005540
2,week5,mean,RandomForestClassifier,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.497234,0.007802,0.519015,0.006099,0.507871,0.006095,0.580781,0.005666,0.562142,0.007154
3,week5,mean,SVC,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.603720,0.005435,0.461337,0.008549,0.522961,0.005375,0.658374,0.003657,0.633681,0.003001
4,week5,mean,KNeighborsClassifier,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.528808,0.006211,0.481911,0.015703,0.504186,0.010263,0.607044,0.004862,0.587579,0.005123
5,week7,mean,DecisionTreeClassifier,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.491793,0.005177,0.487489,0.007957,0.489581,0.003693,0.550527,0.007118,0.567598,0.004494
6,week7,mean,GaussianNB,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.521048,0.006071,0.765341,0.007303,0.619980,0.005590,0.686661,0.002753,0.600837,0.007424
7,week7,mean,RandomForestClassifier,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.500931,0.006226,0.508733,0.015419,0.504735,0.009660,0.595951,0.005861,0.575403,0.005416
8,week7,mean,SVC,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.622916,0.006791,0.443156,0.009209,0.517843,0.007559,0.669973,0.004017,0.648976,0.004161
9,week7,mean,KNeighborsClassifier,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.537662,0.008422,0.485295,0.014355,0.510111,0.011563,0.621929,0.007119,0.603563,0.006735


## 6. 결과 저장

 `paper_replication_baseline_results.csv`로 저장한다



In [24]:
results_path = REPORT_TABLE_DIR / "paper_replication_baseline_results.csv"

cutoff_order = ["week5", "week7", "week10"]
results_to_save = results.copy()
results_to_save["cutoff"] = pd.Categorical(
    results_to_save["cutoff"], categories=cutoff_order, ordered=True
)
results_to_save = results_to_save.sort_values(["cutoff", "model"]).reset_index(drop=True)
results_to_save["cutoff"] = results_to_save["cutoff"].astype(str)

results_to_save.to_csv(results_path, index=False, encoding="utf-8")

print(f"Saved full results: {results_path}")

results_to_save


Saved full results: C:\Users\osca0\Dev\studyDataMining\teamproject\reports\tables\model_results\paper_replication_baseline_results.csv


,cutoff,strategy,model,n_rows,positive_class,target,features,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std
0,week5,mean,DecisionTreeClassifier,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.491442,0.005863,0.484020,0.005825,0.487701,0.005741,0.544346,0.005252,0.557370,0.005025
1,week5,mean,GaussianNB,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.522189,0.004557,0.764229,0.005064,0.620422,0.003309,0.671995,0.003399,0.592938,0.005540
2,week5,mean,KNeighborsClassifier,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.528808,0.006211,0.481911,0.015703,0.504186,0.010263,0.607044,0.004862,0.587579,0.005123
3,week5,mean,RandomForestClassifier,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.497234,0.007802,0.519015,0.006099,0.507871,0.006095,0.580781,0.005666,0.562142,0.007154
4,week5,mean,SVC,27244,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.603720,0.005435,0.461337,0.008549,0.522961,0.005375,0.658374,0.003657,0.633681,0.003001
5,week7,mean,DecisionTreeClassifier,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.491793,0.005177,0.487489,0.007957,0.489581,0.003693,0.550527,0.007118,0.567598,0.004494
6,week7,mean,GaussianNB,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.521048,0.006071,0.765341,0.007303,0.619980,0.005590,0.686661,0.002753,0.600837,0.007424
7,week7,mean,KNeighborsClassifier,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.537662,0.008422,0.485295,0.014355,0.510111,0.011563,0.621929,0.007119,0.603563,0.006735
8,week7,mean,RandomForestClassifier,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.500931,0.006226,0.508733,0.015419,0.504735,0.009660,0.595951,0.005861,0.575403,0.005416
9,week7,mean,SVC,26776,at_risk,target_at_risk,"total_click_until_cutoff_mean_strategy,active_...",0.622916,0.006791,0.443156,0.009209,0.517843,0.007559,0.669973,0.004017,0.648976,0.004161


## 7. 해석 시 주의사항

이 노트북은 엄격한 두-feature 논문 재현형 베이스라인이다. 성능이 낮거나 특정 모델이 불안정하더라도 이는 더 많은 특징을 추가하기 전의 기준선으로 해석해야 한다.

특히 이 실험은 인구통계, assessment, module/presentation, activity_type-specific click feature, missing flag를 의도적으로 제외했기 때문에 최종 모델 후보가 아니다. VLE 중복 처리는 Stage 5 baseline 산출물에서 mean strategy로 확정 적용되었으므로 이 노트북에서는 sum/max 비교를 반복하지 않는다.
